# Retrieval Evaluation — MS-ADS RAG Chatbot

Compares five ChromaDB collections on **9 scored queries** (Q01–Q09), with and without LLM-based query routing.  
Q10 (dress code) is an edge case with no ground truth — shown in manual inspection only.

| Metric | Definition |
|--------|------------|
| **Hit@5** | 1 if any of the top-5 retrieved chunks comes from an expected page, else 0 |
| **MRR@5** | 1/rank of the first hit in top-5; 0 if none |

**Collections**: `msads_minilm_size_512` · `msads_minilm_size_900` · `msads_bgesmall_size_900` · `msads_bgesmall_size_1536` · `msads_bgesmall_size_1900`

**Routing**: `route_query()` (DeepSeek-chat, max_tokens=5) classifies each query as `in_person` / `online` / `general` and applies a ChromaDB `$ne` filter to exclude wrong-scope chunks.

## 1  Imports and Config

In [13]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from openai import OpenAI

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.retrieval.retriever import Retriever
from src.retrieval.query_router import route_query

load_dotenv(PROJECT_ROOT / ".env")
deepseek_client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
)

COLLECTIONS = [
    {"name": "msads_minilm_size_512",    "model": "sentence-transformers/all-MiniLM-L6-v2"},
    {"name": "msads_minilm_size_900",    "model": "sentence-transformers/all-MiniLM-L6-v2"},
    {"name": "msads_bgesmall_size_900",  "model": "BAAI/bge-small-en-v1.5"},
    {"name": "msads_bgesmall_size_1536", "model": "BAAI/bge-small-en-v1.5"},
    {"name": "msads_bgesmall_size_1900", "model": "BAAI/bge-small-en-v1.5"},
]
TOP_K = 5

# Collection used for routing comparison (best performer in baseline)
ROUTE_COLLECTION = "msads_bgesmall_size_1536"

## 2  Load Test Set

In [14]:
test_set = json.loads(
    (PROJECT_ROOT / "eval" / "retrieval_test_set.json").read_text(encoding="utf-8")
)

scored   = [q for q in test_set if q["expected_pages"]]   # Q01–Q09
edge     = [q for q in test_set if not q["expected_pages"]]  # Q10

print(f"Total queries : {len(test_set)}")
print(f"  Scored      : {len(scored)}  (have expected_pages)")
print(f"  Edge case   : {len(edge)}   (no expected_pages — excluded from metrics)")
for q in test_set:
    marker = "[edge]" if not q["expected_pages"] else ""
    print(f"  {q['id']} [{q['difficulty']:6}] {q['query'][:60]}  {marker}")

Total queries : 10
  Scored      : 9  (have expected_pages)
  Edge case   : 1   (no expected_pages — excluded from metrics)
  Q01 [easy  ] What are the core courses required in the in-person MS in Ap  
  Q02 [easy  ] What's the application deadline for the online program?  
  Q03 [medium] How much does the entire program cost?  
  Q04 [easy  ] Tell me about the capstone project experience.  
  Q05 [medium] Which faculty members have research interests in machine lea  
  Q06 [hard  ] What's the difference between the in-person and online progr  
  Q07 [medium] What kind of background do I need to apply?  
  Q08 [medium] Where do graduates of this program typically work?  
  Q09 [easy  ] Are there any scholarships available?  
  Q10 [edge  ] What is the dress code for the program?  [edge]


## 3  Metric Helpers

In [15]:
def hit_at_k(results: list[dict], expected_pages: list[str], k: int = 5) -> int:
    """1 if any of the top-k results comes from an expected page, else 0."""
    top_k_urls = {r["source_url"] for r in results[:k]}
    return int(any(url in top_k_urls for url in expected_pages))


def rr_at_k(results: list[dict], expected_pages: list[str], k: int = 5) -> float:
    """Reciprocal rank of the first hit in top-k; 0.0 if none."""
    for rank, r in enumerate(results[:k], start=1):
        if r["source_url"] in expected_pages:
            return 1.0 / rank
    return 0.0

## 4  Initialize Retrievers

> **Slow cell** — loads sentence-transformer models into GPU memory. Run once per session.

In [16]:
retrievers: dict[str, Retriever] = {}
for spec in COLLECTIONS:
    print(f"Loading {spec['name']} ...")
    retrievers[spec["name"]] = Retriever(spec["name"], spec["model"])
    print(f"  -> ready ({retrievers[spec['name']]._collection.count()} chunks)")

print("\nAll retrievers ready.")

Loading msads_minilm_size_512 ...
  -> ready (753 chunks)
Loading msads_minilm_size_900 ...
  -> ready (561 chunks)
Loading msads_bgesmall_size_900 ...
  -> ready (561 chunks)
Loading msads_bgesmall_size_1536 ...
  -> ready (447 chunks)
Loading msads_bgesmall_size_1900 ...
  -> ready (441 chunks)

All retrievers ready.


## 5  Baseline: Run All Queries × Collections (No Routing)

> **Run once.** Results cached in `raw_results`; all later cells read from it.

In [17]:
# raw_results[collection_name][query_id] = list[dict]
raw_results: dict[str, dict[str, list[dict]]] = {}

for spec in COLLECTIONS:
    cname = spec["name"]
    raw_results[cname] = {}
    r = retrievers[cname]
    for q in test_set:
        raw_results[cname][q["id"]] = r.retrieve(q["query"], top_k=TOP_K)
    print(f"  {cname}: {len(test_set)} queries done")

print("\nBaseline retrieval complete.")

  msads_minilm_size_512: 10 queries done
  msads_minilm_size_900: 10 queries done
  msads_bgesmall_size_900: 10 queries done
  msads_bgesmall_size_1536: 10 queries done
  msads_bgesmall_size_1900: 10 queries done

Baseline retrieval complete.


## 5b  Routed: Run All Queries with LLM Query Router

Uses `route_query()` (DeepSeek-chat) to classify each query and apply a `program_type` `$ne` filter.  
Routing is only tested on the best-performing collection (`ROUTE_COLLECTION`).

**Filter logic**:
- `in_person` → `{"program_type": {"$ne": "online"}}` — exclude online-only chunks
- `online`    → `{"program_type": {"$ne": "in_person"}}` — exclude in-person-only chunks
- `general`   → `None` — no filter, same as baseline

In [18]:
# route_labels[query_id] = "in_person" | "online" | "general"
# route_filters[query_id] = where dict or None
# routed_results[query_id] = list[dict]

route_labels:  dict[str, str]        = {}
route_filters: dict[str, dict | None] = {}
routed_results: dict[str, list[dict]] = {}

r = retrievers[ROUTE_COLLECTION]

for q in test_set:
    where = route_query(q["query"], deepseek_client)
    # derive label from the filter for display
    if where is None:
        label = "general"
    elif where.get("program_type", {}).get("$ne") == "online":
        label = "in_person"
    else:
        label = "online"

    route_labels[q["id"]]  = label
    route_filters[q["id"]] = where
    routed_results[q["id"]] = r.retrieve(q["query"], top_k=TOP_K, where=where)
    print(f"  {q['id']}  label={label:10s}  filter={str(where)[:45]}")

print(f"\nRouted retrieval complete ({ROUTE_COLLECTION}).")

  Q01  label=in_person   filter={'program_type': {'$ne': 'online'}}
  Q02  label=online      filter={'program_type': {'$ne': 'in_person'}}
  Q03  label=general     filter=None
  Q04  label=general     filter=None
  Q05  label=general     filter=None
  Q06  label=general     filter=None
  Q07  label=general     filter=None
  Q08  label=general     filter=None
  Q09  label=general     filter=None
  Q10  label=general     filter=None

Routed retrieval complete (msads_bgesmall_size_1536).


---
## 6  Summary Table — Hit@5 and MRR@5 per Collection (Baseline)

Primary decision-support output.  
Q10 excluded (no expected pages).

In [19]:
rows = []
for spec in COLLECTIONS:
    cname = spec["name"]
    for q in scored:
        results = raw_results[cname][q["id"]]
        rows.append({
            "collection": cname,
            "query_id": q["id"],
            "difficulty": q["difficulty"],
            "type": q["type"],
            "hit5": hit_at_k(results, q["expected_pages"]),
            "rr5":  rr_at_k(results, q["expected_pages"]),
        })

df = pd.DataFrame(rows)

summary = (
    df.groupby("collection")[["hit5", "rr5"]]
    .mean()
    .rename(columns={"hit5": "Hit@5", "rr5": "MRR@5"})
    .sort_values("MRR@5", ascending=False)
    .round(3)
)
print("=== Baseline: Hit@5 and MRR@5 (averaged over 9 scored queries) ===")
display(summary)

=== Baseline: Hit@5 and MRR@5 (averaged over 9 scored queries) ===


,Hit@5,MRR@5
collection,,
msads_bgesmall_size_1536,1.000,0.833
msads_bgesmall_size_900,1.000,0.833
msads_minilm_size_512,0.889,0.833
msads_bgesmall_size_1900,1.000,0.815
msads_minilm_size_900,0.889,0.815


## 6b  Routing Impact — Before vs After (on best collection)

Compares baseline vs routed retrieval for the `ROUTE_COLLECTION`.  
Only queries where routing applies a filter (label ≠ `general`) can change.

In [20]:
compare_rows = []
for q in scored:
    baseline = raw_results[ROUTE_COLLECTION][q["id"]]
    routed   = routed_results[q["id"]]
    label    = route_labels[q["id"]]
    compare_rows.append({
        "query_id":      q["id"],
        "difficulty":    q["difficulty"],
        "route_label":   label,
        "hit_baseline":  hit_at_k(baseline, q["expected_pages"]),
        "hit_routed":    hit_at_k(routed,   q["expected_pages"]),
        "mrr_baseline":  round(rr_at_k(baseline, q["expected_pages"]), 3),
        "mrr_routed":    round(rr_at_k(routed,   q["expected_pages"]), 3),
    })

df_cmp = pd.DataFrame(compare_rows).set_index("query_id")
df_cmp["mrr_delta"] = (df_cmp["mrr_routed"] - df_cmp["mrr_baseline"]).round(3)

print(f"=== Routing impact on {ROUTE_COLLECTION} ===")
display(df_cmp)

print("\n--- Averages ---")
print(f"Baseline  Hit@5={df_cmp['hit_baseline'].mean():.3f}  MRR@5={df_cmp['mrr_baseline'].mean():.3f}")
print(f"Routed    Hit@5={df_cmp['hit_routed'].mean():.3f}  MRR@5={df_cmp['mrr_routed'].mean():.3f}")
print(f"Delta MRR@5  : {df_cmp['mrr_delta'].mean():.3f}")

=== Routing impact on msads_bgesmall_size_1536 ===


,difficulty,route_label,hit_baseline,hit_routed,mrr_baseline,mrr_routed,mrr_delta
query_id,,,,,,,
Q01,easy,in_person,1,1,0.5,1.0,0.5
Q02,easy,online,1,1,1.0,1.0,0.0
Q03,medium,general,1,1,1.0,1.0,0.0
Q04,easy,general,1,1,1.0,1.0,0.0
Q05,medium,general,1,1,0.5,0.5,0.0
Q06,hard,general,1,1,0.5,0.5,0.0
Q07,medium,general,1,1,1.0,1.0,0.0
Q08,medium,general,1,1,1.0,1.0,0.0
Q09,easy,general,1,1,1.0,1.0,0.0



--- Averages ---
Baseline  Hit@5=1.000  MRR@5=0.833
Routed    Hit@5=1.000  MRR@5=0.889
Delta MRR@5  : 0.056


## 7  Per-Difficulty Breakdown (Baseline)

In [21]:
breakdown = (
    df.groupby(["collection", "difficulty"])[["hit5", "rr5"]]
    .mean()
    .rename(columns={"hit5": "Hit@5", "rr5": "MRR@5"})
    .round(3)
)
print("=== Per-difficulty breakdown ===")
display(breakdown)

=== Per-difficulty breakdown ===


Hit@5  MRR@5
collection               difficulty              
msads_bgesmall_size_1536 easy         1.00  0.875
                         hard         1.00  0.500
                         medium       1.00  0.875
msads_bgesmall_size_1900 easy         1.00  0.875
                         hard         1.00  0.500
                         medium       1.00  0.833
msads_bgesmall_size_900  easy         1.00  0.875
                         hard         1.00  0.500
                         medium       1.00  0.875
msads_minilm_size_512    easy         1.00  0.875
                         hard         1.00  1.000
                         medium       0.75  0.750
msads_minilm_size_900    easy         1.00  0.833
                         hard         1.00  1.000
                         medium       0.75  0.750

## 8  Per-Query Score Table (Baseline)

In [22]:
pivot = df.pivot_table(
    index=["query_id", "difficulty"],
    columns="collection",
    values=["hit5", "rr5"],
    aggfunc="first",
).round(3)

print("=== Per-query scores (1=hit, 0=miss; rr=reciprocal rank) ===")
display(pivot)

=== Per-query scores (1=hit, 0=miss; rr=reciprocal rank) ===


hit5                           \
collection          msads_bgesmall_size_1536 msads_bgesmall_size_1900   
query_id difficulty                                                     
Q01      easy                              1                        1   
Q02      easy                              1                        1   
Q03      medium                            1                        1   
Q04      easy                              1                        1   
Q05      medium                            1                        1   
Q06      hard                              1                        1   
Q07      medium                            1                        1   
Q08      medium                            1                        1   
Q09      easy                              1                        1   

                                                                   \
collection          msads_bgesmall_size_900 msads_minilm_size_512   
query_id difficulty                                                 
Q01      easy                             1                     1   
Q02      easy                             1                     1   
Q03      medium                           1                     1   
Q04      easy                             1                     1   
Q05      medium                           1                     1   
Q06      hard                             1                     1   
Q07      medium                           1                     1   
Q08      medium                           1                     0   
Q09      easy                             1                     1   

                                                               rr5  \
collection          msads_minilm_size_900 msads_bgesmall_size_1536   
query_id difficulty                                                  
Q01      easy                           1                      0.5   
Q02      easy                           1                      1.0   
Q03      medium                         1                      1.0   
Q04      easy                           1                      1.0   
Q05      medium                         1                      0.5   
Q06      hard                           1                      0.5   
Q07      medium                         1                      1.0   
Q08      medium                         0                      1.0   
Q09      easy                           1                      1.0   

                                                                      \
collection          msads_bgesmall_size_1900 msads_bgesmall_size_900   
query_id difficulty                                                    
Q01      easy                          0.500                     0.5   
Q02      easy                          1.000                     1.0   
Q03      medium                        1.000                     1.0   
Q04      easy                          1.000                     1.0   
Q05      medium                        0.333                     0.5   
Q06      hard                          0.500                     0.5   
Q07      medium                        1.000                     1.0   
Q08      medium                        1.000                     1.0   
Q09      easy                          1.000                     1.0   

                                                                 
collection          msads_minilm_size_512 msads_minilm_size_900  
query_id difficulty                                              
Q01      easy                         0.5                 0.333  
Q02      easy                         1.0                 1.000  
Q03      medium                       1.0                 1.000  
Q04      easy                         1.0                 1.000  
Q05      medium                       1.0                 1.000  
Q06      hard                         1.0                 1.000  
Q07      medium                   

---
## 9  Manual Inspection — Top-5 Chunks per Query

Toggle **Routing** to compare baseline vs. routed results for the selected collection.  
`[✓]` = chunk's `source_url` is in `expected_pages` for that query.  
Text truncated to 800 chars. Full text in `raw_results` / `routed_results`.

In [23]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import textwrap

clear_output(wait=True)

TRUNC = 800
WRAP  = 90
query_map = {q["id"]: q for q in test_set}

query_options = [(f"{q['id']}  ({q['difficulty']})", q["id"]) for q in test_set]
coll_options  = [spec["name"] for spec in COLLECTIONS]

query_selector = widgets.ToggleButtons(
    options=query_options,
    value=test_set[0]["id"],
    description="",
    button_style="",
    style={"button_width": "90px"},
    layout=widgets.Layout(flex_wrap="wrap"),
)

coll_selector = widgets.Dropdown(
    options=coll_options,
    value=ROUTE_COLLECTION,
    description="Collection:",
    layout=widgets.Layout(width="380px"),
)

routing_toggle = widgets.Checkbox(
    value=False,
    description="Use routing filter",
    layout=widgets.Layout(width="200px"),
)

out = widgets.Output(layout=widgets.Layout(
    border="1px solid #ccc",
    padding="10px",
    max_height="700px",
    overflow_y="auto",
))

_last = [None, None, None]  # [query_id, coll_name, use_routing]


def render(query_id: str, coll_name: str, use_routing: bool) -> None:
    _last[:] = [query_id, coll_name, use_routing]
    SEP = "─" * 68
    with out:
        out.clear_output(wait=True)
        q = query_map[query_id]
        expected_set = set(q["expected_pages"])

        label  = route_labels.get(query_id, "n/a")
        filt   = route_filters.get(query_id)
        mode   = f"ROUTED (label={label}, filter={filt})" if use_routing else "BASELINE (no filter)"

        print(f"[{q['id']}]  {q['query']}")
        print(f"difficulty: {q['difficulty']} | type: {q['type']}")
        print(f"mode      : {mode}")
        if expected_set:
            for url in sorted(expected_set):
                print(f"  expected: {url}")
        else:
            print("  expected: (none — edge case)")

        # pick result list
        if use_routing and coll_name == ROUTE_COLLECTION:
            results = routed_results[query_id][:5]
        elif use_routing:
            # apply filter on-the-fly for other collections
            results = retrievers[coll_name].retrieve(
                q["query"], top_k=TOP_K, where=filt
            )[:5]
        else:
            results = raw_results[coll_name][query_id][:5]

        print()
        print(SEP)
        print(f"  Collection: {coll_name}")
        print(SEP)
        for i, r in enumerate(results, start=1):
            marker = "✓" if r["source_url"] in expected_set else " "
            text_flat  = r["text"][:TRUNC].replace("\n", " ")
            text_lines = textwrap.wrap(text_flat, width=WRAP)
            print(f"  [{marker}] Rank {i}  dist={r['distance']:.4f}  program_type={r.get('program_type', '?')}")
            print(f"       url        : {r['source_url']}")
            print(f"       section    : {r.get('section_breadcrumb', r.get('section', ''))}")
            print(f"       content_type: {r.get('content_type', '')}")
            indent = "       text       : "
            cont   = " " * len(indent)
            if text_lines:
                print(f"{indent}{text_lines[0]}")
                for line in text_lines[1:]:
                    print(f"{cont}{line}")
            print()


def on_change(change):
    if [query_selector.value, coll_selector.value, routing_toggle.value] != _last:
        render(query_selector.value, coll_selector.value, routing_toggle.value)


controls = widgets.HBox([coll_selector, routing_toggle])
display(widgets.VBox([query_selector, controls, out]))
_last[:] = [query_selector.value, coll_selector.value, routing_toggle.value]
query_selector.observe(on_change, names="value")
coll_selector.observe(on_change, names="value")
routing_toggle.observe(on_change, names="value")
render(query_selector.value, coll_selector.value, routing_toggle.value)